# Speculative Decoding - 실습 코드 2: Speculative Decoding 원리 구현 (순수 PyTorch)

- Tutorial ID: `expand-speculative-decoding`
- Tutorial: Speculative Decoding
- Section ID: `expand-speculative-decoding-code-2`
- Section: 실습 코드 2: Speculative Decoding 원리 구현 (순수 PyTorch)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: Speculative Decoding 원리 구현 (순수 PyTorch)
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

# ── 작은 모델 (Draft)과 큰 모델 (Target) ──
class TinyLM(nn.Module):
    """간소화된 언어모델"""
        def __init__(self, vocab_size=1000, d_model=128, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                                d_model=d_model, nhead=4,
                dim_feedforward=d_model*4,
                batch_first=True,
            ) for _ in range(n_layers)
        ])
        self.head = nn.Linear(d_model, vocab_size)
    
        def forward(self, x):
                x = self.embed(x)
                for layer in self.layers:
                        x = layer(x)
        return self.head(x)
    
    @torch.no_grad()
        def get_logprobs(self, x):
        """시퀀스의 각 위치에서 로그 확률 반환"""
                logits = self.forward(x)
        return F.log_softmax(logits, dim=-1)

# ── Speculative Decoding 구현 ──
def speculative_decode(
    draft_model, target_model, 
    prompt_ids, max_new_tokens=100,
    speculative_length=5, temperature=1.0,
):
    """Speculative Decoding 알고리즘
    
    1. Draft 모델이 K개 토큰 자기회귀 생성
    2. Target 모델이 K+1개 토큰을 한 번에 검증
    3. 일치하면 수락, 불일치 시 첫 불일치 위치에서 재샘플링
    """
        tokens = prompt_ids.clone()
    total_draft = 0
    total_accepted = 0
    total_steps = 0
    
        while tokens.size(1) < prompt_ids.size(1) + max_new_tokens:
        total_steps += 1
        
        # ── Step 1: Draft 모델이 K개 토큰 제안 ──
        draft_tokens = []
                draft_logprobs = []
        current = tokens
        
                for k in range(speculative_length):
                        logprobs = draft_model.get_logprobs(current)
            next_logprob = logprobs[:, -1, :]  # 마지막 위치
            
            if temperature > 0:
                                probs = F.softmax(next_logprob / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = next_logprob.argmax(dim=-1, keepdim=True)
            
            draft_tokens.append(next_token)
            draft_logprobs.append(F.log_softmax(next_logprob, dim=-1).gather(1, next_token))
            current = torch.cat([current, next_token], dim=1)
            total_draft += 1
        
        # ── Step 2: Target 모델이 병렬 검증 ──
        # Draft 토큰을 포함한 전체 시퀀스를 한 번에 Target에 통과
        draft_seq = torch.cat([tokens] + draft_tokens, dim=1)
                target_logprobs = target_model.get_logprobs(draft_seq)
        
        # ── Step 3: 수락/거절 판정 ──
        n_accepted = 0
                for k in range(speculative_length):
            pos = tokens.size(1) - 1 + k  # 검증할 위치
            
            # Target의 해당 토큰 로그 확률
            target_prob = target_logprobs[:, pos, :].exp()
            draft_prob = draft_logprobs[k].exp()
            
            # 수락 확률: min(1, p_target / p_draft)
            acceptance_ratio = target_prob.gather(1, draft_tokens[k]) / (draft_prob + 1e-10)
            
            if torch.rand(1).item() < acceptance_ratio.item():
                n_accepted += 1
                total_accepted += 1
            else:
                # 거절: Target 분포에서 재샘플링
                if temperature > 0:
                    resampled = torch.multinomial(target_prob, num_samples=1)
                else:
                    resampled = target_prob.argmax(dim=-1, keepdim=True)
                
                # 수락된 토큰 + 재샘플링 토큰만 추가
                accepted_tokens = torch.cat(draft_tokens[:n_accepted] + [resampled], dim=1)
                                tokens = torch.cat([tokens, accepted_tokens], dim=1)
                break
        else:
            # 모두 수락! + Target이 제안하는 다음 토큰 1개 추가
            bonus_token_pos = draft_seq.size(1) - 1
                        bonus_probs = target_logprobs[:, bonus_token_pos, :].exp()
            if temperature > 0:
                bonus_token = torch.multinomial(bonus_probs, num_samples=1)
            else:
                bonus_token = bonus_probs.argmax(dim=-1, keepdim=True)
                        tokens = torch.cat([tokens, 
                torch.cat(draft_tokens + [bonus_token], dim=1)], dim=1)
            total_accepted += speculative_length
    
    acceptance_rate = total_accepted / max(total_draft, 1)
    return tokens, acceptance_rate, total_steps

# ── 자기회귀 디코딩 (비교용) ──
def autoregressive_decode(model, prompt_ids, max_new_tokens=100, temperature=1.0):
    """표준 자기회귀 디코딩"""
        tokens = prompt_ids.clone()
    with torch.no_grad():
                for _ in range(max_new_tokens):
                        logprobs = model.get_logprobs(tokens)
            next_prob = F.softmax(logprobs[:, -1, :] / max(temperature, 0.01), dim=-1)
            next_token = torch.multinomial(next_prob, num_samples=1)
                        tokens = torch.cat([tokens, next_token], dim=1)
    return tokens

# ── 실험 ──
print("=== Speculative Decoding 실험 ===\n")

vocab_size = 1000
draft_model = TinyLM(vocab_size, d_model=64, n_layers=1)   # 작은 모델
target_model = TinyLM(vocab_size, d_model=128, n_layers=4)  # 큰 모델

draft_params = sum(p.numel() for p in draft_model.parameters())
target_params = sum(p.numel() for p in target_model.parameters())
print(f"Draft 모델:  {draft_params:,} 파라미터")
print(f"Target 모델: {target_params:,} 파라미터 ({target_params/draft_params:.1f}x)\n")

prompt = torch.randint(0, vocab_size, (1, 10))

# Speculative Decoding
start = time.time()
spec_tokens, acc_rate, steps = speculative_decode(
    draft_model, target_model, prompt,
    max_new_tokens=50, speculative_length=5
)
spec_time = time.time() - start

print(f"Speculative Decoding:")
print(f"  생성 토큰: {spec_tokens.size(1) - 10}")
print(f"  수락률: {acc_rate*100:.1f}%")
print(f"  Target 호출: {steps}회 (vs 자기회귀: {spec_tokens.size(1)-10}회)")
print(f"  시간: {spec_time:.3f}초")

# Auto-regressive 비교
start = time.time()
ar_tokens = autoregressive_decode(target_model, prompt, max_new_tokens=50)
ar_time = time.time() - start

print(f"\nAutoregressive Decoding:")
print(f"  시간: {ar_time:.3f}초")
speedup = ar_time / spec_time if spec_time > 0 else 0
print(f"\n⚡ 속도 향상: {speedup:.1f}x")